# 参数优化教程

本教程介绍 open-xquant 的参数优化模块 `oxq.optimize`，覆盖策略参数搜索和样本外验证的完整流程：

1. **ParameterSet** — 定义参数搜索空间 + 约束条件
2. **GridSearch** — 网格搜索，穷举所有参数组合
3. **WalkForward** — 滚动/锚定前推分析，评估样本外表现
4. **TimeSeriesCV** — 时间序列交叉验证

### 为什么需要参数优化？

> *Every trading system is in some form an optimization.* — Tomasini

参数优化的目标是找到最符合假设和目标的参数组合。但要注意：

- **过拟合是最大风险** — 在样本内表现出色的参数未必在样本外有效
- **稳定区域比最优点更重要** — 相邻参数组合应有相似的表现（Tomasini & Jaekle 2009）
- **Walk-forward 是标准验证方法** — 用滚动窗口在真正的样本外数据上评估参数

### 前置条件

需要已下载 AAPL 行情数据：
```bash
python -c "from oxq.data import YFinanceDownloader; \
    YFinanceDownloader().download('AAPL', '2020-01-01', '2024-12-31')"
```

In [ ]:
# 下载数据（如已有可跳过）
from oxq.data import YFinanceDownloader

downloader = YFinanceDownloader()
path = downloader.download("AAPL", start="2020-01-01", end="2024-12-31")
print(f"数据已保存到: {path}")

---
## 1. 构建基础策略

我们用 SMA 均线交叉策略作为贯穿全文的示例。策略有两个关键参数：
- `sma_fast` 的 `period`（短期均线周期）
- `sma_slow` 的 `period`（长期均线周期）

这两个参数的选择直接影响策略表现。我们将用 `oxq.optimize` 来系统地搜索最优参数。

In [ ]:
from oxq.core import Engine, Strategy
from oxq.data import LocalMarketDataProvider
from oxq.indicators import SMA
from oxq.rules import EntryRule, ExitRule
from oxq.signals import Crossover
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse

# 基础策略：SMA(10) / SMA(50) 交叉
strategy = Strategy(
    name="sma_crossover",
    hypothesis="短期均线上穿长期均线时买入，死叉时卖出",
    objectives={
        "total_return": {"min": 0.05},
        "sharpe_ratio": {"min": 0.5},
        "max_drawdown": {"max": -0.20},
    },
    universe=StaticUniverse(("AAPL",)),
    indicators={
        "sma_fast": (SMA(), {"period": 10}),
        "sma_slow": (SMA(), {"period": 50}),
    },
    signals={
        "golden_cross": (Crossover(), {"fast": "sma_fast", "slow": "sma_slow"}),
    },
    entry_rules=[EntryRule(signal="golden_cross", shares=100)],
    exit_rules=[ExitRule(fast="sma_fast", slow="sma_slow")],
)

# 先用默认参数跑一次回测作为基准
result = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
)

print(f"基准策略 SMA(10, 50):")
print(f"  总收益率:     {result.total_return():.2%}")
print(f"  Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"  最大回撤:     {result.max_drawdown():.2%}")
print(f"  交易次数:     {len(result.trades)}")

---
## 2. ParameterSet — 定义参数搜索空间

`ParameterSet` 声明要搜索哪些参数、取值范围，以及参数之间的约束。

关键概念：
- `add(component, param, values)` — 注册一个参数的候选值
- `add_constraint(expr)` — 添加参数间约束（如快线周期必须小于慢线）
- `grid()` — 生成所有满足约束的参数组合

In [ ]:
from oxq.optimize import ParameterSet

paramset = ParameterSet("sma_tuning")

# 快线周期：5, 10, 15, 20, 25
paramset.add("sma_fast", "period", values=range(5, 30, 5))

# 慢线周期：20, 30, 40, 50, 60, 70, 80
paramset.add("sma_slow", "period", values=range(20, 90, 10))

# 约束：快线周期必须小于慢线周期（否则没有意义）
paramset.add_constraint("sma_fast.period < sma_slow.period")

print(f"总组合数（约束前）: {paramset.total_combinations}")
print(f"有效组合数（约束后）: {len(paramset.grid())}")
print(f"过滤掉的无效组合:   {paramset.total_combinations - len(paramset.grid())}")
print()
print("前 5 个有效组合：")
for combo in paramset.grid()[:5]:
    print(f"  sma_fast.period={combo['sma_fast']['period']}, "
          f"sma_slow.period={combo['sma_slow']['period']}")

约束表达式支持 `<`, `>`, `<=`, `>=`, `==`, `!=`，操作数可以是 `component.param` 引用或数字字面量。

约束是可序列化的字符串，而非 lambda —— 这保证参数空间定义可以导出为 JSON/YAML，也让 AI Agent 能直接读写。

---
## 3. GridSearch — 网格搜索

`GridSearch` 对每个有效参数组合运行一次完整回测，收集所有结果。

核心设计：
- `broker_factory` 传入一个工厂函数（如 `lambda: SimBroker()`），确保每次回测用全新的 broker，无状态泄漏
- `metric` 指定优化目标（默认 `sharpe_ratio`）
- `metric_direction` 控制优化方向（maximize/minimize），内置合理默认值

In [ ]:
from oxq.optimize import GridSearch

search_result = GridSearch(paramset).run(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    metric="sharpe_ratio",
)

print(f"测试了 {len(search_result.all_results)} 个参数组合")
print()

# 最优参数
best = search_result.best
print("最优参数组合：")
print(f"  sma_fast.period = {best.params['sma_fast']['period']}")
print(f"  sma_slow.period = {best.params['sma_slow']['period']}")
print(f"  Sharpe Ratio    = {best.metric_value:.4f}")
print(f"  总收益率         = {best.run_result.total_return():.2%}")
print(f"  最大回撤         = {best.run_result.max_drawdown():.2%}")

### 查看 Top 5 参数组合

不要只看最优点——要关注参数空间中是否存在"稳定区域"（stable region）。如果 top 5 的参数相近且表现相似，说明结果是稳健的。如果 top 5 参数散布在完全不同的区域，则过拟合风险很高。

In [ ]:
# Top 5 参数组合
print(f"{'Rank':>4}  {'Fast':>6}  {'Slow':>6}  {'Sharpe':>8}  {'Return':>8}  {'MaxDD':>8}  {'Trades':>6}")
print("-" * 56)

for i, trial in enumerate(search_result.top_n(5), 1):
    rr = trial.run_result
    print(
        f"{i:>4}  "
        f"{trial.params['sma_fast']['period']:>6}  "
        f"{trial.params['sma_slow']['period']:>6}  "
        f"{trial.metric_value:>8.4f}  "
        f"{rr.total_return():>7.2%}  "
        f"{rr.max_drawdown():>7.2%}  "
        f"{len(rr.trades):>6}"
    )

### 用 DataFrame 分析全部结果

`to_dataframe()` 将所有参数组合展平为 DataFrame，便于进一步分析和可视化：

In [ ]:
df = search_result.to_dataframe()
print(f"DataFrame shape: {df.shape}")
print(f"列: {list(df.columns)}")
print()

# 按 Sharpe 排序显示
print(df.sort_values("metric_value", ascending=False)[
    ["sma_fast.period", "sma_slow.period", "sharpe_ratio", "total_return", "max_drawdown", "num_trades"]
].to_string(index=False))

### 自定义 metric

除了内置的 `sharpe_ratio`、`total_return` 等，还可以传入自定义 callable 作为优化目标：

In [ ]:
# 自定义 metric: Sharpe / |MaxDD| 的比值（风险调整后的夏普）
custom_result = GridSearch(paramset).run(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    metric=lambda r: r.sharpe_ratio() / max(abs(r.max_drawdown()), 0.001),
    metric_direction="maximize",
)

best_custom = custom_result.best
print("自定义 metric (Sharpe / |MaxDD|) 最优参数：")
print(f"  sma_fast.period = {best_custom.params['sma_fast']['period']}")
print(f"  sma_slow.period = {best_custom.params['sma_slow']['period']}")
print(f"  metric_value    = {best_custom.metric_value:.4f}")
print(f"  Sharpe Ratio    = {best_custom.run_result.sharpe_ratio():.4f}")
print(f"  最大回撤         = {best_custom.run_result.max_drawdown():.2%}")

---
## 4. WalkForward — 前推分析

> *Walk forward analysis guards against data mining bias.* — Peterson (2017)

GridSearch 的结果是**样本内（in-sample）**表现，可能过拟合。Walk-forward 将数据切分为多个滚动的训练/测试窗口：

**Rolling（滚动窗口）**:
```
|--train--|--test--|
      |--train--|--test--|
            |--train--|--test--|
```

**Anchored（锚定窗口）**:
```
|----train----|--test--|
|------train------|--test--|
|--------train--------|--test--|
```

每个窗口内：在训练集上 GridSearch 找最优参数，然后在测试集上评估。测试集的表现是真正的**样本外（OOS）**结果。

In [ ]:
from oxq.optimize import WalkForward

# Rolling Walk-Forward: 2年训练 + 6个月测试
wf = WalkForward(
    paramset,
    train_period="2Y",
    test_period="6M",
    anchored=False,
)

wf_result = wf.run(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    metric="sharpe_ratio",
)

print(f"Walk-Forward 窗口数: {len(wf_result.windows)}")
print()

# 每个窗口的详情
print(f"{'Window':>6}  {'Train':>23}  {'Test':>23}  {'IS Sharpe':>10}  {'OOS Sharpe':>10}  {'Fast':>5}  {'Slow':>5}")
print("-" * 95)
for i, w in enumerate(wf_result.windows, 1):
    oos_sharpe = w.oos_result.sharpe_ratio()
    print(
        f"{i:>6}  "
        f"{w.train_start} ~ {w.train_end}  "
        f"{w.test_start} ~ {w.test_end}  "
        f"{w.in_sample_metric:>10.4f}  "
        f"{oos_sharpe:>10.4f}  "
        f"{w.best_params['sma_fast']['period']:>5}  "
        f"{w.best_params['sma_slow']['period']:>5}"
    )

### OOS 汇总指标和退化分析

`deterioration()` 对比样本内和样本外的平均表现。如果退化过大（如 IS Sharpe 1.5 → OOS Sharpe 0.3），说明参数过拟合严重。

In [ ]:
# OOS 汇总指标
print("Walk-Forward OOS 汇总：")
print(f"  OOS 总收益率:     {wf_result.oos_total_return():.2%}")
print(f"  OOS Sharpe Ratio: {wf_result.oos_sharpe_ratio():.4f}")
print(f"  OOS 最大回撤:     {wf_result.oos_max_drawdown():.2%}")
print()

# IS vs OOS 退化分析
det = wf_result.deterioration()
print("IS vs OOS 退化（负数 = OOS 更差）：")
for metric_name, ratio in det.items():
    print(f"  {metric_name}: {ratio:+.2%}")

### Anchored Walk-Forward 对比

Anchored 模式从固定起点开始，训练窗口逐步扩大。适合数据有限或需要利用更多历史信息的场景：

In [ ]:
# Anchored Walk-Forward
wf_anchored = WalkForward(
    paramset,
    train_period="2Y",
    test_period="6M",
    anchored=True,
)

wf_anchored_result = wf_anchored.run(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    metric="sharpe_ratio",
)

# 对比 Rolling vs Anchored
print(f"{'':>20}  {'Rolling':>12}  {'Anchored':>12}")
print("-" * 48)
print(f"{'窗口数':>20}  {len(wf_result.windows):>12}  {len(wf_anchored_result.windows):>12}")
print(f"{'OOS 总收益率':>20}  {wf_result.oos_total_return():>11.2%}  {wf_anchored_result.oos_total_return():>11.2%}")
print(f"{'OOS Sharpe':>20}  {wf_result.oos_sharpe_ratio():>12.4f}  {wf_anchored_result.oos_sharpe_ratio():>12.4f}")
print(f"{'OOS MaxDD':>20}  {wf_result.oos_max_drawdown():>11.2%}  {wf_anchored_result.oos_max_drawdown():>11.2%}")

### WalkForward DataFrame

`to_dataframe()` 输出每个窗口的详细信息，便于分析参数稳定性：

In [ ]:
wf_df = wf_result.to_dataframe()
print(wf_df[[
    "train_start", "train_end", "test_start", "test_end",
    "sma_fast.period", "sma_slow.period",
    "in_sample_metric", "oos_sharpe_ratio", "oos_total_return",
]].to_string(index=False))

---
## 5. TimeSeriesCV — 时间序列交叉验证

`TimeSeriesCV` 提供两种功能：
1. **`split()`** — 纯函数，生成 train/test 日期分割（不运行回测）
2. **`cross_validate()`** — 在每个 fold 上运行策略（可选参数优化）

与 sklearn 的 `TimeSeriesSplit` 类似，但为量化回测定制，支持 embargo 间隔防止自相关泄漏。

In [ ]:
from oxq.optimize import TimeSeriesCV

# 查看 split() 生成的日期分割
cv = TimeSeriesCV(n_splits=4, embargo_days=5, expanding=True)
splits = cv.split("2020-01-01", "2024-12-31")

print(f"Expanding CV — {len(splits)} 个 folds:")
print(f"{'Fold':>4}  {'Train':>23}  {'Embargo':>3}  {'Test':>23}")
print("-" * 65)
for i, s in enumerate(splits, 1):
    print(f"{i:>4}  {s.train_start} ~ {s.train_end}  {'5d':>3}  {s.test_start} ~ {s.test_end}")

### 无参数优化的交叉验证

不传 `paramset` 时，`cross_validate()` 直接用策略原始参数在每个 fold 的 test 期上评估——验证策略在不同时间段的鲁棒性：

In [ ]:
# 无参数优化：验证固定参数在不同时间段的表现
cv_result = cv.cross_validate(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    metric="sharpe_ratio",
)

print(f"OOS Sharpe 均值: {cv_result.mean_oos_metric():.4f}")
print(f"OOS Sharpe 标准差: {cv_result.std_oos_metric():.4f}")
print()

# 每个 fold 的 OOS 表现
print(f"{'Fold':>4}  {'Test Period':>23}  {'OOS Sharpe':>10}  {'OOS Return':>10}  {'OOS MaxDD':>10}")
print("-" * 65)
for i, sr in enumerate(cv_result.splits, 1):
    rr = sr.oos_result
    print(
        f"{i:>4}  {sr.split.test_start} ~ {sr.split.test_end}  "
        f"{rr.sharpe_ratio():>10.4f}  "
        f"{rr.total_return():>9.2%}  "
        f"{rr.max_drawdown():>9.2%}"
    )

### 带参数优化的交叉验证

传入 `paramset` 后，每个 fold 会在训练集上做 GridSearch，然后用最优参数在测试集上评估：

In [ ]:
# 带参数优化的交叉验证
cv_opt_result = cv.cross_validate(
    strategy=strategy,
    market=LocalMarketDataProvider(),
    broker_factory=lambda: SimBroker(),
    start="2020-01-01",
    end="2024-12-31",
    paramset=paramset,
    metric="sharpe_ratio",
)

print(f"OOS Sharpe 均值（带优化）: {cv_opt_result.mean_oos_metric():.4f}")
print(f"OOS Sharpe 标准差:         {cv_opt_result.std_oos_metric():.4f}")
print()

print(f"{'Fold':>4}  {'IS Sharpe':>10}  {'OOS Sharpe':>10}  {'Fast':>5}  {'Slow':>5}")
print("-" * 45)
for i, sr in enumerate(cv_opt_result.splits, 1):
    rr = sr.oos_result
    print(
        f"{i:>4}  "
        f"{sr.in_sample_metric:>10.4f}  "
        f"{rr.sharpe_ratio():>10.4f}  "
        f"{sr.best_params['sma_fast']['period']:>5}  "
        f"{sr.best_params['sma_slow']['period']:>5}"
    )

---
## 6. 总结对比

将三种方法的结果放在一起，评估哪种方式给出最稳健的参数：

In [ ]:
print("=" * 60)
print("方法对比总结")
print("=" * 60)
print()

# 1. 基准（默认参数）
print(f"{'方法':>20}  {'Sharpe':>8}  {'Return':>8}  {'MaxDD':>8}")
print("-" * 50)
print(f"{'基准 SMA(10,50)':>20}  {result.sharpe_ratio():>8.4f}  {result.total_return():>7.2%}  {result.max_drawdown():>7.2%}")

# 2. GridSearch（全样本内）
print(f"{'GridSearch 最优':>20}  {best.metric_value:>8.4f}  {best.run_result.total_return():>7.2%}  {best.run_result.max_drawdown():>7.2%}")

# 3. Walk-Forward OOS
print(f"{'WF Rolling OOS':>20}  {wf_result.oos_sharpe_ratio():>8.4f}  {wf_result.oos_total_return():>7.2%}  {wf_result.oos_max_drawdown():>7.2%}")
print(f"{'WF Anchored OOS':>20}  {wf_anchored_result.oos_sharpe_ratio():>8.4f}  {wf_anchored_result.oos_total_return():>7.2%}  {wf_anchored_result.oos_max_drawdown():>7.2%}")

# 4. CV
print(f"{'CV 固定参数 (mean)':>20}  {cv_result.mean_oos_metric():>8.4f}")
print(f"{'CV 优化参数 (mean)':>20}  {cv_opt_result.mean_oos_metric():>8.4f}")

print()
print("注意：GridSearch 的结果是样本内表现，通常高于真实样本外。")
print("Walk-Forward 和 TimeSeriesCV 的 OOS 结果更接近实际可预期的表现。")

---
## 小结

本教程覆盖了参数优化模块的核心概念：

| 组件 | 职责 | 关键方法 |
|------|------|----------|
| `ParameterSet` | 定义参数搜索空间 + 约束 | `add()`, `add_constraint()`, `grid()` |
| `GridSearch` | 穷举参数组合回测 | `run()` → `SearchResult` |
| `WalkForward` | 滚动/锚定前推分析 | `run()` → `WalkForwardResult` |
| `TimeSeriesCV` | 时间序列交叉验证 | `split()`, `cross_validate()` → `CVResult` |

**参数优化的红线规则**（Peterson 2017）：

1. **永远不要只看样本内结果** — GridSearch 的最优 ≠ 实际可预期的表现
2. **寻找稳定区域，而非最优点** — 相邻参数应有相似表现
3. **参数组合 > 100 时需要统计校正** — 未来用 Deflated Sharpe Ratio
4. **Walk-Forward 窗口应覆盖至少 2 个市场周期**
5. **不要在参数优化后添加新规则** — 这是 Rule Burden，本质上是 data snooping
6. **Walk-Forward 应谨慎使用** — 多次运行并挑选结果同样是数据偷窥

**下一步**：
- `oxq.optimize.validation` 未来将增加统计检验（Deflated Sharpe、Profit Hurdle 等）
- 结合 `engine.run(run_through="signal")` 可以在不执行交易规则的情况下单独优化信号参数